<a href="https://colab.research.google.com/github/guitorte/audio/blob/ccr-8be874fb-rdbsnf/voice-conversion/Voice_Conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎤 Voice Conversion: Seed-VC and RVC

Replaces the singer in your songs with another voice, and keeps the instrumental untouched.

```
song ─► BS-RoFormer ─┬─ vocal ──► Seed-VC  or  RVC ──► level-matched vocal ─┐
                     └─ instrumental ───────────────────────────────────────┴─► song__voice__engine.wav
```

| | **Seed-VC** | **RVC** (via Applio) |
|---|---|---|
| What you need | one 10–30 s clip of the target voice | a trained voice model (`.pth`, optionally `.index`) |
| Training | none (zero-shot) | ~10+ min of clean recordings → cell 4 trains one (1–3 h on a T4) |
| Strengths | instant; try any voice | usually closer to the voice once trained; fast conversion |

**Put your files in Drive** inside your songs folder (default `áudio`):

```
áudio/
├── my song.mp3              ← songs to convert
└── voices/                  ← created by cell 1
    ├── maria.wav            ← Seed-VC reference clip (a song is fine: the vocal is isolated)
    ├── maria.pth            ← RVC model (+ maria.index) — trained here or brought in
    └── maria/               ← RVC training recordings (cell 4)
```

Results go to `áudio/converted/`: the full mix, plus a `_vocals.wav` with just the new vocal.

⚖️ Only convert voices you have the right to use, such as your own or a collaborator's who has agreed.

`Runtime ▸ Change runtime type ▸ T4 GPU`, then run cell 1, and after it whichever cell you need.

In [ ]:
#@title 1 · Connect Google Drive and list songs and voices
DRIVE_FOLDER = "áudio"  #@param {type:"string"}

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

REPO, BRANCH = "/content/audio", "ccr-8be874fb-rdbsnf"
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                       "https://github.com/guitorte/audio.git", REPO]).returncode:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/guitorte/audio.git", REPO], check=True)
sys.path.insert(0, f"{REPO}/voice-conversion")
import vc
U = vc.U

FOLDER = U.find_folder(DRIVE_FOLDER)
FILES = U.list_audio(FOLDER)
print(f"📁 {FOLDER}  ({len(FILES)} songs)")
for i, f in enumerate(FILES, 1):
    print(f"{i:3d}. {f.name}")
print()
print(vc.describe(vc.scan_voices(FOLDER)))
RESULTS = []

In [ ]:
#@title 2 · Seed-VC: convert with a reference clip (no training)
SONGS = "1"  #@param {type:"string"}
#@markdown `all`, numbers (`1, 3-4`) or parts of names.
REFERENCE = "1"  #@param {type:"string"}
#@markdown Number or name of a clip in `voices/`.
SEMITONES = 0  #@param {type:"slider", min:-12, max:12, step:1}
#@markdown Shift the melody, e.g. `+12` for a male song sung by a female voice, `-12` the other way.
DIFFUSION_STEPS = 30  #@param {type:"slider", min:10, max:60, step:5}
#@markdown 30–50 is recommended for singing. More steps take longer.
SINGING_MODEL = True  #@param {type:"boolean"}
#@markdown Untick for spoken word (22 kHz speech model).
SEPARATE_VOCALS = True  #@param {type:"boolean"}
#@markdown Untick only if the song files are already a cappella.
ISOLATE_REFERENCE_VOCAL = True  #@param {type:"boolean"}
VOCAL_GAIN_DB = 0.0  #@param {type:"slider", min:-6, max:6, step:0.5}
OVERWRITE = False  #@param {type:"boolean"}

opts = vc.Options(separate=SEPARATE_VOCALS, vocal_gain_db=VOCAL_GAIN_DB, overwrite=OVERWRITE,
                  singing=SINGING_MODEL, semitones=SEMITONES, steps=DIFFUSION_STEPS,
                  isolate_reference=ISOLATE_REFERENCE_VOCAL)
RESULTS = vc.convert(U.select_files(FILES, SONGS), FOLDER, "seedvc", REFERENCE, opts)
for r in RESULTS:
    print(f"\n{os.path.basename(r.source)} → {r.output}", *[f"   · {n}" for n in r.notes + [r.skipped] if n], sep="\n")

In [ ]:
#@title 3 · RVC: convert with a voice model
SONGS = "1"  #@param {type:"string"}
MODEL = ""  #@param {type:"string"}
#@markdown Name of a `.pth` in `voices/` (without the extension).
PITCH = 0  #@param {type:"slider", min:-12, max:12, step:1}
INDEX_RATE = 0.5  #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown How strongly to pull towards the training voice's timbre (needs the `.index`). Higher = more like the voice, sometimes less clear.
PROTECT = 0.33  #@param {type:"slider", min:0, max:0.5, step:0.01}
#@markdown Protects breaths and consonants from artefacts. Lower = stronger protection.
SEPARATE_VOCALS = True  #@param {type:"boolean"}
VOCAL_GAIN_DB = 0.0  #@param {type:"slider", min:-6, max:6, step:0.5}
OVERWRITE = False  #@param {type:"boolean"}

opts = vc.Options(separate=SEPARATE_VOCALS, vocal_gain_db=VOCAL_GAIN_DB, overwrite=OVERWRITE,
                  semitones=PITCH, index_rate=INDEX_RATE, protect=PROTECT)
RESULTS = vc.convert(U.select_files(FILES, SONGS), FOLDER, "rvc", MODEL, opts)
for r in RESULTS:
    print(f"\n{os.path.basename(r.source)} → {r.output}", *[f"   · {n}" for n in r.notes + [r.skipped] if n], sep="\n")

In [ ]:
#@title 4 · RVC: train a voice model from your recordings
DATASET = ""  #@param {type:"string"}
#@markdown Name of a subfolder of `voices/` holding the recordings (10+ min in total works best; songs are fine).
MODEL_NAME = ""  #@param {type:"string"}
#@markdown Empty = same as the folder.
ISOLATE_VOCALS = True  #@param {type:"boolean"}
#@markdown Strip instruments from each recording first. Untick for clean a cappella takes.
EPOCHS = 200  #@param {type:"integer"}
SAVE_EVERY = 10  #@param {type:"integer"}
BATCH_SIZE = 8  #@param {type:"integer"}
SAMPLE_RATE = "40000"  #@param ["32000", "40000", "48000"]

#@markdown ⏱ Training takes hours. Checkpoints are mirrored to `voices/.training/` every few minutes, so if Colab disconnects, re-run this cell with the same settings and it resumes. When it finishes, `voices/<name>.pth` and `.index` are ready for cell 3.
topts = vc.TrainOptions(isolate_vocals=ISOLATE_VOCALS, epochs=EPOCHS, save_every=SAVE_EVERY,
                        batch_size=BATCH_SIZE, sample_rate=int(SAMPLE_RATE))
vc.train(FOLDER, DATASET, MODEL_NAME, topts)
print()
print(vc.describe(vc.scan_voices(FOLDER)))

In [ ]:
#@title 5 · Listen: original vs converted (last run of cell 2 or 3)
START_AT_SECONDS = 45  #@param {type:"number"}
CLIP_SECONDS = 30  #@param {type:"number"}

import numpy as np
from IPython.display import Audio, Markdown, display

def excerpt(path):
    return np.clip(U.excerpt(path, START_AT_SECONDS, CLIP_SECONDS, 44100).T, -1, 1)

for r in RESULTS:
    if not r.output:
        continue
    display(Markdown(f"### {os.path.basename(r.output)}"))
    for title, path in (("Original", r.source), ("Converted", r.output), ("Converted vocal only", r.vocals)):
        if path:
            display(Markdown(f"**{title}**")); display(Audio(excerpt(path), rate=44100, normalize=False))

### Tips
- **Pitch range matters most.** If the target voice is much higher or lower than the original singer, use `SEMITONES` / `PITCH` (±12 = one octave), or the result strains or sounds unnatural.
- **Seed-VC reference:** a clean, dry clip of the voice singing works best. A full song is fine because the vocal is isolated, and the loudest 25 s are used.
- **RVC training data:** 10–30 min of a single voice, dry, no backing vocals or heavy reverb. Quality of data beats quantity.
- **Mostly hear the instrumental?** The vocal split works best on mixes with a clear lead vocal. Backing vocals stay in the instrumental and keep the original voice.